In [1]:
import os

for name in os.listdir("/kaggle/input/datasets/zlemglsmklkaya"):
    print(name)

caai-revision-splits
caai-models-nb1-complete


In [2]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SPLITS_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
NB1_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb1-complete"
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Credit card
cc_dir = os.path.join(SPLITS_ROOT, "credit_card")
cc_train = pd.read_parquet(os.path.join(cc_dir, "train.parquet"))
cc_id    = pd.read_parquet(os.path.join(cc_dir, "id.parquet"))
cc_id2   = pd.read_parquet(os.path.join(cc_dir, "id2.parquet"))
cc_ood   = pd.read_parquet(os.path.join(cc_dir, "ood.parquet"))
with open(os.path.join(cc_dir, "config.json")) as f:
    cc_config = json.load(f)
CC_FEATURES = cc_config["model_features"]
print(f"\nCredit card: Train={cc_train.shape}, ID={cc_id.shape}, ID2={cc_id2.shape}, OOD={cc_ood.shape}")

# CIC-IDS2017, both scenarios
cic_scenarios = {}
for scen in ["scenario_T", "scenario_A"]:
    d = os.path.join(SPLITS_ROOT, "cic_ids2017", scen)
    cic_scenarios[scen] = {
        "train": pd.read_parquet(os.path.join(d, "train.parquet")),
        "id":    pd.read_parquet(os.path.join(d, "id.parquet")),
        "id2":   pd.read_parquet(os.path.join(d, "id2.parquet")),
        "ood":   pd.read_parquet(os.path.join(d, "ood.parquet")),
    }
with open(os.path.join(SPLITS_ROOT, "cic_ids2017", "config.json")) as f:
    cic_config = json.load(f)
CIC_FEATURES = cic_config["model_features"]
print(f"CIC-IDS2017 features ({len(CIC_FEATURES)})")
for scen, d in cic_scenarios.items():
    print(f"{scen}: Train={d['train'].shape}, ID={d['id'].shape}, ID2={d['id2'].shape}, OOD={d['ood'].shape}")

# Rebuild fit/val (deterministic — reproduces NB1's exact partitions)
def make_fit_val_split(train_df, val_frac=0.10, seed=SEED):
    shuffled = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_val = int(len(shuffled) * val_frac)
    return shuffled.iloc[n_val:].copy(), shuffled.iloc[:n_val].copy()

fit_cc, val_cc = make_fit_val_split(cc_train)
fit_T, val_T = make_fit_val_split(cic_scenarios["scenario_T"]["train"])
fit_A, val_A = make_fit_val_split(cic_scenarios["scenario_A"]["train"])
print(f"\nfit/val sizes: CC=({len(fit_cc)},{len(val_cc)})  T=({len(fit_T)},{len(val_T)})  A=({len(fit_A)},{len(val_A)})")

Device: cuda
GPU: Tesla T4

Credit card: Train=(100774, 31), ID=(21731, 31), ID2=(21731, 31), OOD=(139490, 31)
CIC-IDS2017 features (67)
scenario_T: Train=(910016, 82), ID=(296457, 82), ID2=(296457, 82), OOD=(1017868, 82)
scenario_A: Train=(1190196, 82), ID=(357584, 82), ID2=(357585, 82), OOD=(615433, 82)

fit/val sizes: CC=(90697,10077)  T=(819015,91001)  A=(1071177,119019)


In [3]:
class DeepSVDDNet(nn.Module):
    """Encoder-only network, NO bias terms anywhere — required to prevent
    trivial collapse to a constant output (Ruff et al., 2018)."""
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h, bias=False), nn.ReLU()]
            prev = h
        layers += [nn.Linear(prev, latent_dim, bias=False)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def init_center(model, X_init, device, eps=0.1):
    """Center = mean embedding over an initial batch, nudged away from 0
    on any near-zero dimension (a center at the origin combined with
    bias-free layers is itself a degenerate solution)."""
    model.eval()
    with torch.no_grad():
        outputs = model(X_init.to(device))
        c = outputs.mean(dim=0)
    c[(torch.abs(c) < eps) & (c < 0)] = -eps
    c[(torch.abs(c) < eps) & (c >= 0)] = eps
    return c


def train_deep_svdd(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = DeepSVDDNet(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    # Center initialized from a sample of fit data, BEFORE training starts
    init_sample = X_fit[torch.randperm(len(X_fit))[:min(10000, len(X_fit))]]
    c = init_center(model, init_sample, device)
    print(f"  Center initialized: mean|c|={c.abs().mean().item():.4f}, min|c|={c.abs().min().item():.4f}")

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit),
        batch_size=train_config["batch_size"], shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_embedding_var": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            z = model(batch)
            loss = torch.mean(torch.sum((z - c) ** 2, dim=1))
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_loss = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            z_val = model(X_val)
            val_loss = torch.mean(torch.sum((z_val - c) ** 2, dim=1)).item()
            val_embedding_var = z_val.var(dim=0).mean().item()  # collapse diagnostic

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_embedding_var"].append(val_embedding_var)

        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}  "
                  f"embedding_var={val_embedding_var:.6f}")

        if val_loss < best_val_loss - train_config["min_delta"]:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, c, history, best_val_loss, epoch


def compute_svdd_score(model, c, df, feature_cols, device):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        z = model(X)
        dist = torch.sum((z - c) ** 2, dim=1).cpu().numpy()
    return dist


def score_split_svdd(model, c, df, feature_cols, device, split_name):
    scores = compute_svdd_score(model, c, df, feature_cols, device)
    out = pd.DataFrame({"row_id": np.arange(len(df)), "Class": df["Class"].values, "svdd_distance": scores})
    benign = out[out.Class == 0].svdd_distance
    anomaly = out[out.Class == 1].svdd_distance
    print(f"{split_name}: benign mean={benign.mean():.4f} median={benign.median():.4f} | "
          f"anomaly mean={anomaly.mean():.4f} median={anomaly.median():.4f} "
          f"(n_benign={len(benign)}, n_anomaly={len(anomaly)})")
    return out

print("Deep SVDD classes and functions defined.")

Deep SVDD classes and functions defined.


In [4]:
import time

ARCH_CONFIG_SVDD_CC_PROBE = {"hidden_dims": [20, 14], "latent_dim": 7}
probe_config = {
    "learning_rate": 1e-3, "batch_size": 256, "max_epochs": 5,
    "patience": 999, "min_delta": 1e-5,
}

t0 = time.time()
_, _, probe_history, _, _ = train_deep_svdd(
    fit_df=fit_cc, val_df=val_cc, feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_SVDD_CC_PROBE, train_config=probe_config, device=DEVICE, log_every=1,
)
elapsed = time.time() - t0
print(f"\n5 epochs took {elapsed:.1f}s -> ~{elapsed/5:.1f}s/epoch")

  Center initialized: mean|c|=0.1000, min|c|=0.1000
  Epoch   1  train_loss=0.011401  val_loss=0.003210  embedding_var=0.000451
  Epoch   2  train_loss=0.002054  val_loss=0.001087  embedding_var=0.000154
  Epoch   3  train_loss=0.000645  val_loss=0.000434  embedding_var=0.000062
  Epoch   4  train_loss=0.000274  val_loss=0.000196  embedding_var=0.000028
  Epoch   5  train_loss=0.000134  val_loss=0.000113  embedding_var=0.000016

5 epochs took 8.9s -> ~1.8s/epoch


In [5]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []; prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        enc_layers += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)
        dec_layers = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)
    def forward(self, x):
        return self.decoder(self.encoder(x))


def build_svdd_from_pretrained_ae(ae_state_dict, input_dim, hidden_dims, latent_dim, device):
    """Copy AE encoder weights into a bias-free Deep SVDD net (drop biases,
    keep weight matrices) — the standard Ruff et al. warm-start procedure."""
    svdd_net = DeepSVDDNet(input_dim, hidden_dims, latent_dim).to(device)
    ae_layers = [k for k in ae_state_dict.keys() if k.startswith("encoder") and "weight" in k]
    svdd_layers = [k for k in svdd_net.state_dict().keys() if "weight" in k]
    assert len(ae_layers) == len(svdd_layers), f"Layer count mismatch: {len(ae_layers)} vs {len(svdd_layers)}"
    new_state = svdd_net.state_dict()
    for ae_key, svdd_key in zip(ae_layers, svdd_layers):
        new_state[svdd_key] = ae_state_dict[ae_key].clone()
    svdd_net.load_state_dict(new_state)
    return svdd_net


def train_deep_svdd_warmstart(fit_df, val_df, feature_cols, arch_config, train_config,
                                pretrained_ae_state, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = build_svdd_from_pretrained_ae(
        pretrained_ae_state, input_dim, arch_config["hidden_dims"], arch_config["latent_dim"], device
    )

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    init_sample = X_fit[torch.randperm(len(X_fit))[:min(10000, len(X_fit))]]
    c = init_center(model, init_sample, device)
    print(f"  Center initialized (from pretrained encoder): mean|c|={c.abs().mean().item():.4f}")

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit), batch_size=train_config["batch_size"], shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"],
                                   weight_decay=train_config["weight_decay"])

    best_val_loss = float("inf"); best_state = None; epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_embedding_var": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train(); epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            z = model(batch)
            loss = torch.mean(torch.sum((z - c) ** 2, dim=1))
            loss.backward(); optimizer.step()
            epoch_losses.append(loss.item())
        train_loss = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            z_val = model(X_val)
            val_loss = torch.mean(torch.sum((z_val - c) ** 2, dim=1)).item()
            val_embedding_var = z_val.var(dim=0).mean().item()

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["val_embedding_var"].append(val_embedding_var)

        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}  embedding_var={val_embedding_var:.6f}")

        if val_loss < best_val_loss - train_config["min_delta"]:
            best_val_loss = val_loss; best_state = {k: v.clone() for k, v in model.state_dict().items()}; epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose: print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, c, history, best_val_loss, epoch

print("Warm-start training function defined.")

Warm-start training function defined.


In [7]:
NB1_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb1-complete"
for root, dirs, files in os.walk(NB1_ROOT):
    depth = root.replace(NB1_ROOT, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    for f in files:
        print(f"{indent}  {f}")

caai-models-nb1-complete/
  vae_credit_card/
    config.json
    scores_id2.parquet
    scores_ood.parquet
    scores_id.parquet
    model.pt
  ae_cic_scenario_A/
    config.json
    scores_id2.parquet
    scores_ood.parquet
    scores_id.parquet
    model.pt
  vae_cic_scenario_A/
    config.json
    scores_id2.parquet
    scores_ood.parquet
    scores_id.parquet
    model.pt
  vae_cic_scenario_T/
    config.json
    scores_id2.parquet
    scores_ood.parquet
    scores_id.parquet
    model.pt


In [8]:
AE_CC_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits-new"
ae_state_cc = torch.load(os.path.join(AE_CC_ROOT, "ae_credit_card", "model.pt"), map_location=DEVICE)
print("Loaded ae_credit_card from caai-revision-splits-new")

Loaded ae_credit_card from caai-revision-splits-new


In [9]:
probe_config_v2 = {
    "learning_rate": 1e-3, "batch_size": 256, "max_epochs": 5,
    "patience": 999, "min_delta": 1e-5, "weight_decay": 1e-5,
}

_, _, probe_history_v2, _, _ = train_deep_svdd_warmstart(
    fit_df=fit_cc, val_df=val_cc, feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_SVDD_CC_PROBE, train_config=probe_config_v2,
    pretrained_ae_state=ae_state_cc, device=DEVICE, log_every=1,
)

  Center initialized (from pretrained encoder): mean|c|=0.9220
  Epoch   1  train_loss=9.300474  val_loss=2.479235  embedding_var=0.321603
  Epoch   2  train_loss=1.384207  val_loss=0.839020  embedding_var=0.116803
  Epoch   3  train_loss=0.620363  val_loss=0.485273  embedding_var=0.068394
  Epoch   4  train_loss=0.381417  val_loss=0.326945  embedding_var=0.046197
  Epoch   5  train_loss=0.260121  val_loss=0.233316  embedding_var=0.032888


In [10]:
probe_config_v3 = {
    "learning_rate": 1e-3, "batch_size": 256, "max_epochs": 10,
    "patience": 999, "min_delta": 1e-5, "weight_decay": 1e-3,
}

_, _, probe_history_v3, _, _ = train_deep_svdd_warmstart(
    fit_df=fit_cc, val_df=val_cc, feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_SVDD_CC_PROBE, train_config=probe_config_v3,
    pretrained_ae_state=ae_state_cc, device=DEVICE, log_every=1,
)

  Center initialized (from pretrained encoder): mean|c|=0.9181
  Epoch   1  train_loss=9.213062  val_loss=2.498394  embedding_var=0.329258
  Epoch   2  train_loss=1.398613  val_loss=0.841653  embedding_var=0.116993
  Epoch   3  train_loss=0.612939  val_loss=0.480672  embedding_var=0.067440
  Epoch   4  train_loss=0.380046  val_loss=0.323714  embedding_var=0.045722
  Epoch   5  train_loss=0.262106  val_loss=0.229963  embedding_var=0.032604
  Epoch   6  train_loss=0.188649  val_loss=0.169717  embedding_var=0.024093
  Epoch   7  train_loss=0.140608  val_loss=0.127577  embedding_var=0.018086
  Epoch   8  train_loss=0.106875  val_loss=0.099968  embedding_var=0.014191
  Epoch   9  train_loss=0.083469  val_loss=0.080586  embedding_var=0.011502
  Epoch  10  train_loss=0.065696  val_loss=0.063120  embedding_var=0.008989


In [11]:
def train_deep_svdd_soft_boundary(fit_df, val_df, feature_cols, arch_config, train_config,
                                    pretrained_ae_state, device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    model = build_svdd_from_pretrained_ae(
        pretrained_ae_state, input_dim, arch_config["hidden_dims"], arch_config["latent_dim"], device
    )

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    init_sample = X_fit[torch.randperm(len(X_fit))[:min(10000, len(X_fit))]]
    c = init_center(model, init_sample, device)

    # R starts as a learnable scalar, initialized to a small positive value
    R = torch.tensor(1.0, requires_grad=True, device=device)
    nu = train_config["nu"]  # expected fraction of points allowed outside the boundary

    print(f"  Center initialized: mean|c|={c.abs().mean().item():.4f}, initial R={R.item():.4f}, nu={nu}")

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit), batch_size=train_config["batch_size"], shuffle=True
    )
    optimizer = torch.optim.Adam(
        list(model.parameters()) + [R], lr=train_config["learning_rate"], weight_decay=train_config["weight_decay"]
    )

    best_val_loss = float("inf"); best_state = None; best_R = None; epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_embedding_var": [], "R": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train(); epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            z = model(batch)
            dist = torch.sum((z - c) ** 2, dim=1)
            scores = dist - R ** 2
            loss = R ** 2 + (1 / nu) * torch.mean(torch.clamp(scores, min=0))
            loss.backward()
            optimizer.step()
            # R must stay non-negative
            with torch.no_grad():
                R.clamp_(min=0)
            epoch_losses.append(loss.item())
        train_loss = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            z_val = model(X_val)
            dist_val = torch.sum((z_val - c) ** 2, dim=1)
            scores_val = dist_val - R ** 2
            val_loss = (R ** 2 + (1 / nu) * torch.mean(torch.clamp(scores_val, min=0))).item()
            val_embedding_var = z_val.var(dim=0).mean().item()

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["val_embedding_var"].append(val_embedding_var); history["R"].append(R.item())

        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}  "
                  f"embedding_var={val_embedding_var:.6f}  R={R.item():.4f}")

        if val_loss < best_val_loss - train_config["min_delta"]:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_R = R.item()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose: print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, c, best_R, history, best_val_loss, epoch


probe_config_soft = {
    "learning_rate": 1e-3, "batch_size": 256, "max_epochs": 15,
    "patience": 999, "min_delta": 1e-6, "weight_decay": 1e-5,
    "nu": 0.1,  # assume ~10% of training data could be borderline/noisy — standard default
}

_, _, _, probe_history_soft, _, _ = train_deep_svdd_soft_boundary(
    fit_df=fit_cc, val_df=val_cc, feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_SVDD_CC_PROBE, train_config=probe_config_soft,
    pretrained_ae_state=ae_state_cc, device=DEVICE, log_every=1,
)

  Center initialized: mean|c|=0.9290, initial R=1.0000, nu=0.1
  Epoch   1  train_loss=73.652192  val_loss=13.623999  embedding_var=0.289320  R=1.2982
  Epoch   2  train_loss=8.021256  val_loss=5.177986  embedding_var=0.144787  R=1.3578
  Epoch   3  train_loss=4.047918  val_loss=3.425543  embedding_var=0.114154  R=1.3511
  Epoch   4  train_loss=2.985733  val_loss=2.718108  embedding_var=0.094631  R=1.3000
  Epoch   5  train_loss=2.441834  val_loss=2.288575  embedding_var=0.084934  R=1.2272
  Epoch   6  train_loss=2.049589  val_loss=1.908610  embedding_var=0.074250  R=1.1357
  Epoch   7  train_loss=1.741927  val_loss=1.624779  embedding_var=0.065436  R=1.0375
  Epoch   8  train_loss=1.436541  val_loss=1.358174  embedding_var=0.055242  R=0.9371
  Epoch   9  train_loss=1.188279  val_loss=1.168127  embedding_var=0.047567  R=0.8390
  Epoch  10  train_loss=0.986082  val_loss=0.918462  embedding_var=0.038507  R=0.7453
  Epoch  11  train_loss=0.791477  val_loss=0.764442  embedding_var=0.030414

In [13]:
AE_T_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-cic-scenario-t"
ae_state_T = torch.load(os.path.join(AE_T_ROOT, "ae_cic_scenario_T", "model.pt"), map_location=DEVICE)

ARCH_CONFIG_SVDD_CIC_PROBE = {"hidden_dims": [48, 32], "latent_dim": 16}

_, _, _, probe_history_T, _, _ = train_deep_svdd_soft_boundary(
    fit_df=fit_T, val_df=val_T, feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_SVDD_CIC_PROBE, train_config=probe_config_soft,
    pretrained_ae_state=ae_state_T, device=DEVICE, log_every=1,
)

  Center initialized: mean|c|=0.5437, initial R=1.0000, nu=0.1
  Epoch   1  train_loss=15.687399  val_loss=0.690519  embedding_var=0.014663  R=0.5923
  Epoch   2  train_loss=0.381459  val_loss=0.120874  embedding_var=0.001787  R=0.1946
  Epoch   3  train_loss=0.110882  val_loss=0.038993  embedding_var=0.000465  R=0.0983
  Epoch   4  train_loss=0.049970  val_loss=0.022128  embedding_var=0.000203  R=0.0688
  Epoch   5  train_loss=0.027931  val_loss=0.025733  embedding_var=0.000191  R=0.0675
  Epoch   6  train_loss=0.021507  val_loss=0.008921  embedding_var=0.000071  R=0.0302
  Epoch   7  train_loss=0.016274  val_loss=0.006545  embedding_var=0.000046  R=0.0331
  Epoch   8  train_loss=0.008949  val_loss=0.010677  embedding_var=0.000074  R=0.0263
  Epoch   9  train_loss=0.009255  val_loss=0.140310  embedding_var=0.000878  R=0.0371
  Epoch  10  train_loss=0.015244  val_loss=0.005531  embedding_var=0.000038  R=0.0252
  Epoch  11  train_loss=0.015756  val_loss=0.008965  embedding_var=0.000062 

In [14]:
def init_center_coldstart(input_dim, hidden_dims, latent_dim, X_init, device, eps=0.1):
    model = DeepSVDDNet(input_dim, hidden_dims, latent_dim).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(X_init.to(device))
        c = outputs.mean(dim=0)
    c[(torch.abs(c) < eps) & (c < 0)] = -eps
    c[(torch.abs(c) < eps) & (c >= 0)] = eps
    return model, c


def train_deep_svdd_soft_boundary_coldstart(fit_df, val_df, feature_cols, arch_config, train_config,
                                              device, verbose=True, log_every=5):
    input_dim = len(feature_cols)
    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    init_sample = X_fit[torch.randperm(len(X_fit))[:min(10000, len(X_fit))]]
    model, c = init_center_coldstart(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"], init_sample, device)

    R = torch.tensor(1.0, requires_grad=True, device=device)
    nu = train_config["nu"]
    print(f"  Center initialized (cold start): mean|c|={c.abs().mean().item():.4f}, R={R.item():.4f}")

    fit_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_fit), batch_size=train_config["batch_size"], shuffle=True)
    optimizer = torch.optim.Adam(list(model.parameters()) + [R], lr=train_config["learning_rate"], weight_decay=train_config["weight_decay"])

    history = {"train_loss": [], "val_loss": [], "val_embedding_var": [], "R": []}
    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            z = model(batch)
            dist = torch.sum((z - c) ** 2, dim=1)
            scores = dist - R ** 2
            loss = R ** 2 + (1 / nu) * torch.mean(torch.clamp(scores, min=0))
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                R.clamp_(min=0)

        model.eval()
        with torch.no_grad():
            z_val = model(X_val)
            val_embedding_var = z_val.var(dim=0).mean().item()
        history["val_embedding_var"].append(val_embedding_var)
        history["R"].append(R.item())
        if verbose and (epoch == 1 or epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  embedding_var={val_embedding_var:.6f}  R={R.item():.4f}")

    return model, c, R.item(), history


probe_coldstart_config = {"learning_rate": 1e-3, "batch_size": 256, "max_epochs": 15, "weight_decay": 1e-5, "nu": 0.1}

_, _, _, probe_history_cold = train_deep_svdd_soft_boundary_coldstart(
    fit_df=fit_T, val_df=val_T, feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_SVDD_CIC_PROBE, train_config=probe_coldstart_config,
    device=DEVICE, log_every=1,
)

  Center initialized (cold start): mean|c|=0.1000, R=1.0000
  Epoch   1  embedding_var=0.000012  R=0.0149
  Epoch   2  embedding_var=0.000004  R=0.0085
  Epoch   3  embedding_var=0.000005  R=0.0125
  Epoch   4  embedding_var=0.000002  R=0.0062
  Epoch   5  embedding_var=0.000002  R=0.0042
  Epoch   6  embedding_var=0.000002  R=0.0040
  Epoch   7  embedding_var=0.000002  R=0.0046
  Epoch   8  embedding_var=0.000003  R=0.0047
  Epoch   9  embedding_var=0.000006  R=0.0155
  Epoch  10  embedding_var=0.000001  R=0.0059
  Epoch  11  embedding_var=0.000001  R=0.0046
  Epoch  12  embedding_var=0.000005  R=0.0049
  Epoch  13  embedding_var=0.000007  R=0.0114
  Epoch  14  embedding_var=0.000003  R=0.0056
  Epoch  15  embedding_var=0.000001  R=0.0038


In [15]:
def train_deep_svdd_soft_boundary_diagnostic(fit_df, val_df, feature_cols, arch_config, train_config, device, log_every=1):
    input_dim = len(feature_cols)
    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    init_sample = X_fit[torch.randperm(len(X_fit))[:min(10000, len(X_fit))]]
    model, c = init_center_coldstart(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"], init_sample, device)

    R = torch.tensor(1.0, requires_grad=True, device=device)
    nu = train_config["nu"]
    optimizer = torch.optim.Adam(list(model.parameters()) + [R], lr=train_config["learning_rate"])

    fit_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_fit), batch_size=train_config["batch_size"], shuffle=True)

    for epoch in range(1, 4):
        model.train()
        pct_exceeding_R = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            z = model(batch)
            dist = torch.sum((z - c) ** 2, dim=1)
            scores = dist - R ** 2
            pct_exceeding_R.append((scores > 0).float().mean().item())
            loss = R ** 2 + (1 / nu) * torch.mean(torch.clamp(scores, min=0))
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                R.clamp_(min=0)
        print(f"Epoch {epoch}: R={R.item():.4f}, mean %% batches with points exceeding R: {np.mean(pct_exceeding_R)*100:.2f}%%")

train_deep_svdd_soft_boundary_diagnostic(fit_T, val_T, CIC_FEATURES, ARCH_CONFIG_SVDD_CIC_PROBE, probe_coldstart_config, DEVICE)

Epoch 1: R=0.0159, mean %% batches with points exceeding R: 3.93%%
Epoch 2: R=0.0086, mean %% batches with points exceeding R: 10.00%%
Epoch 3: R=0.0076, mean %% batches with points exceeding R: 10.23%%


In [16]:
class DeepSVDDNetNormalized(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h, bias=False), nn.ReLU()]
            prev = h
        layers += [nn.Linear(prev, latent_dim, bias=False)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        z = self.net(x)
        return z / (z.norm(dim=1, keepdim=True) + 1e-8)  # project onto unit hypersphere


def train_deep_svdd_normalized(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True, log_every=1):
    input_dim = len(feature_cols)
    model = DeepSVDDNetNormalized(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        init_sample = X_fit[torch.randperm(len(X_fit))[:min(10000, len(X_fit))]].to(device)
        c = model(init_sample).mean(dim=0)
        c = c / (c.norm() + 1e-8)  # center also lives on the sphere
    print(f"  Center initialized (normalized): |c|={c.norm().item():.4f}")

    fit_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_fit), batch_size=train_config["batch_size"], shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"], weight_decay=train_config["weight_decay"])

    history = {"train_loss": [], "val_embedding_var": []}
    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            z = model(batch)
            loss = torch.mean(torch.sum((z - c) ** 2, dim=1))
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_loss = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            z_val = model(X_val)
            val_embedding_var = z_val.var(dim=0).mean().item()
        history["train_loss"].append(train_loss)
        history["val_embedding_var"].append(val_embedding_var)
        if verbose and (epoch % log_every == 0):
            print(f"  Epoch {epoch:3d}  train_loss={train_loss:.6f}  embedding_var={val_embedding_var:.6f}")

    return model, c, history


probe_norm_config = {"learning_rate": 1e-3, "batch_size": 256, "max_epochs": 15, "weight_decay": 1e-5}

_, _, probe_history_norm = train_deep_svdd_normalized(
    fit_df=fit_T, val_df=val_T, feature_cols=CIC_FEATURES,
    arch_config=ARCH_CONFIG_SVDD_CIC_PROBE, train_config=probe_norm_config, device=DEVICE, log_every=1,
)

  Center initialized (normalized): |c|=1.0000
  Epoch   1  train_loss=0.000893  embedding_var=0.000001
  Epoch   2  train_loss=0.000004  embedding_var=0.000000
  Epoch   3  train_loss=0.000001  embedding_var=0.000000
  Epoch   4  train_loss=0.000001  embedding_var=0.000000
  Epoch   5  train_loss=0.000002  embedding_var=0.000000
  Epoch   6  train_loss=0.000001  embedding_var=0.000000
  Epoch   7  train_loss=0.000001  embedding_var=0.000000
  Epoch   8  train_loss=0.000001  embedding_var=0.000000
  Epoch   9  train_loss=0.000001  embedding_var=0.000000
  Epoch  10  train_loss=0.000001  embedding_var=0.000000
  Epoch  11  train_loss=0.000003  embedding_var=0.000000
  Epoch  12  train_loss=0.000001  embedding_var=0.000000
  Epoch  13  train_loss=0.000001  embedding_var=0.000000
  Epoch  14  train_loss=0.000001  embedding_var=0.000000
  Epoch  15  train_loss=0.000001  embedding_var=0.000000


In [19]:
def build_svdd_from_biasfree_ae(ae_state_dict, input_dim, hidden_dims, latent_dim, device):
    svdd_net = DeepSVDDNet(input_dim, hidden_dims, latent_dim).to(device)
    ae_keys = [k for k in ae_state_dict.keys() if k.startswith("encoder")]
    svdd_keys = list(svdd_net.state_dict().keys())
    assert len(ae_keys) == len(svdd_keys), f"Mismatch: {len(ae_keys)} vs {len(svdd_keys)}"
    new_state = {svdd_k: ae_state_dict[ae_k].clone() for ae_k, svdd_k in zip(ae_keys, svdd_keys)}
    svdd_net.load_state_dict(new_state)
    return svdd_net

svdd_net_T = build_svdd_from_biasfree_ae(biasfree_ae_T.state_dict(), len(CIC_FEATURES),
                                           ARCH_CONFIG_SVDD_CIC_PROBE["hidden_dims"], ARCH_CONFIG_SVDD_CIC_PROBE["latent_dim"], DEVICE)

X_fit_T = torch.tensor(fit_T[CIC_FEATURES].values, dtype=torch.float32)
init_sample = X_fit_T[torch.randperm(len(X_fit_T))[:10000]]
c_T = init_center(svdd_net_T, init_sample, DEVICE)
print(f"True warm-start center: mean|c|={c_T.abs().mean().item():.4f}")

optimizer = torch.optim.Adam(svdd_net_T.parameters(), lr=1e-4, weight_decay=1e-5)
X_val_T = torch.tensor(val_T[CIC_FEATURES].values, dtype=torch.float32).to(DEVICE)
fit_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_fit_T), batch_size=256, shuffle=True)

print("\nTraining Deep SVDD from TRUE bias-free warm-start (lower LR: 1e-4)...")
for epoch in range(1, 16):
    svdd_net_T.train()
    for (batch,) in fit_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        z = svdd_net_T(batch)
        loss = torch.mean(torch.sum((z - c_T) ** 2, dim=1))
        loss.backward()
        optimizer.step()
    svdd_net_T.eval()
    with torch.no_grad():
        z_val = svdd_net_T(X_val_T)
        val_embedding_var = z_val.var(dim=0).mean().item()
    print(f"  Epoch {epoch:3d}  embedding_var={val_embedding_var:.6f}")

True warm-start center: mean|c|=0.6450

Training Deep SVDD from TRUE bias-free warm-start (lower LR: 1e-4)...
  Epoch   1  embedding_var=0.206263
  Epoch   2  embedding_var=0.054325
  Epoch   3  embedding_var=0.016839
  Epoch   4  embedding_var=0.004436
  Epoch   5  embedding_var=0.002748
  Epoch   6  embedding_var=0.001776
  Epoch   7  embedding_var=0.001087
  Epoch   8  embedding_var=0.000806
  Epoch   9  embedding_var=0.000584
  Epoch  10  embedding_var=0.000484
  Epoch  11  embedding_var=0.000516
  Epoch  12  embedding_var=0.000371
  Epoch  13  embedding_var=0.000274
  Epoch  14  embedding_var=0.000258
  Epoch  15  embedding_var=0.000251


In [20]:
print("Continuing training from epoch 16...")
for epoch in range(16, 46):
    svdd_net_T.train()
    for (batch,) in fit_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        z = svdd_net_T(batch)
        loss = torch.mean(torch.sum((z - c_T) ** 2, dim=1))
        loss.backward()
        optimizer.step()
    svdd_net_T.eval()
    with torch.no_grad():
        z_val = svdd_net_T(X_val_T)
        val_embedding_var = z_val.var(dim=0).mean().item()
    if epoch % 2 == 0 or epoch in (16, 45):
        print(f"  Epoch {epoch:3d}  embedding_var={val_embedding_var:.8f}")

Continuing training from epoch 16...
  Epoch  16  embedding_var=0.00020092
  Epoch  18  embedding_var=0.00017087
  Epoch  20  embedding_var=0.00014167
  Epoch  22  embedding_var=0.00011739
  Epoch  24  embedding_var=0.00012595
  Epoch  26  embedding_var=0.00015797
  Epoch  28  embedding_var=0.00007884
  Epoch  30  embedding_var=0.00011365
  Epoch  32  embedding_var=0.00006865
  Epoch  34  embedding_var=0.00005725
  Epoch  36  embedding_var=0.00005505
  Epoch  38  embedding_var=0.00004524
  Epoch  40  embedding_var=0.00004386
  Epoch  42  embedding_var=0.00003666
  Epoch  44  embedding_var=0.00005988
  Epoch  45  embedding_var=0.00003405
